# Load BAFU into Brightway

This notebook loads the BAFU EcoSpold XML files, applies the manual elementary-flow mapping in `flows.csv`, checks mapped flows against the selected ecoinvent biosphere database, writes review reports, and optionally writes the mapped BAFU database into Brightway.

Keep this notebook in the same folder as `utils.py`, `flows.csv`, and `ecospold/`.

## 1. Configure the run

Set the Brightway project and database names. If ecoinvent is already installed in the project, leave `ECOINVENT_USERNAME` and `ECOINVENT_PASSWORD` as `None`. If it is not installed, provide credentials so Brightway can import the configured release.

In [ ]:
from pathlib import Path

import pandas as pd

from utils import BafuLoader

ROOT = Path.cwd()
PROJECT = "test_bafu"
BAFU_DB = "bafu"
ECOINVENT_DB = "ecoinvent-3.12-cutoff"
BIOSPHERE_DB = "ecoinvent-3.12-biosphere"
VERSION = "3.12"
SYSTEM_MODEL = "cutoff"

ECOINVENT_USERNAME = None
ECOINVENT_PASSWORD = None

WRITE_DATABASE = True
OVERWRITE_BAFU = True

## 2. Create and check the loader

`BafuLoader` owns the run state. The check step verifies that the XML folder and mapping CSV exist and that Brightway can be imported.

In [ ]:
loader = BafuLoader(
    project=PROJECT,
    folder=ROOT,
    xml="ecospold",
    csvfile="flows.csv",
    bafu=BAFU_DB,
    ecoinvent=ECOINVENT_DB,
    biosphere=BIOSPHERE_DB,
    version=VERSION,
    system=SYSTEM_MODEL,
    username=ECOINVENT_USERNAME,
    password=ECOINVENT_PASSWORD,
    overwrite=OVERWRITE_BAFU,
)

loader.check()

## 3. Load the XML and mapping table

This cell parses all XML files in `ecospold/` and loads every row from `flows.csv`. It does not write to Brightway yet.

In [ ]:
loader.load()
loader.stats

## 4. Map elementary flows

This step checks the selected ecoinvent databases, imports ecoinvent if needed, applies `flows.csv`, and writes mapping reports to `output/`.

In [ ]:
loader.map()
loader.stats

## 5. Review unresolved flows

If this table has rows, decide whether each BAFU flow should map to an ecoinvent elementary flow. Add the decision to `flows.csv`, record the reason in `README.md`, and rerun from the load step.

In [ ]:
unmapped_file = ROOT / "output" / "unmapped_flows_report.csv"
unmapped = pd.read_csv(unmapped_file, sep=";")
print(f"Unresolved rows: {len(unmapped)}")
unmapped.head(20)

## 6. Write the Brightway database

Run this only after the mapping audit looks acceptable. Flows still unresolved after `flows.csv` is applied are preserved in a Brightway database named `bafu biosphere`, so their exchanges are not silently dropped.

In [ ]:
if WRITE_DATABASE:
    write_stats = loader.write()
else:
    write_stats = {"skipped": True}

write_stats